In [1]:
# === Dependencies required by TA-Mallows ===
import time, itertools, random, pandas as pd, numpy as np

def pairwise_tissue_sets(df: pd.DataFrame):
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    for i, a in enumerate(drugs):
        for j, b in enumerate(drugs):
            if i==j: continue
            wins=set()
            for t in tissues:
                va,vb=df.at[t,a],df.at[t,b]
                if pd.isna(va) or pd.isna(vb): continue
                if va>vb: wins.add(t)
            S[i][j]=wins
    return drugs,tissues,S

def path_intersection_prefix(order_idxs, S, lam):
    if not order_idxs: return [], set(), []
    S_common=None; adj_sets=[]; path=[order_idxs[0]]
    for i,j in zip(order_idxs[:-1], order_idxs[1:]):
        Sab=S[i][j]; S_common=Sab if S_common is None else (S_common & Sab)
        if len(S_common)>=lam:
            path.append(j); adj_sets.append(Sab & S_common)
        else: break
    return path, (S_common if S_common is not None else set()), adj_sets

def tissue_weights_from_agreement(df: pd.DataFrame):
    drugs,tissues,S=pairwise_tissue_sets(df)
    n,m=len(drugs),len(tissues)
    maj={}
    for i in range(n):
        for j in range(n):
            if i==j: continue
            a,b=len(S[i][j]),len(S[j][i])
            if a==b: continue
            maj[(i,j)]=(1 if a>b else 0)
    wt={t:0 for t in tissues}
    for t in tissues:
        score=0; total=0
        for i in range(n):
            for j in range(i+1,n):
                va,vb=df.at[t,drugs[i]],df.at[t,drugs[j]]
                if pd.isna(va) or pd.isna(vb): continue
                total+=1
                if (i,j) in maj or (j,i) in maj:
                    pref=1 if va>vb else (0 if vb>va else None)
                    if pref is not None:
                        if (i,j) in maj and maj[(i,j)]==1 and pref==1: score+=1
                        if (j,i) in maj and maj[(j,i)]==1 and pref==0: score+=1
        wt[t]=score/max(total,1)
    s=sum(wt.values()) or 1.0
    for t in wt: wt[t]*=(len(tissues)/s)
    return wt

def kendall_disagreement(order_idx, df: pd.DataFrame, weights):
    drugs=list(df.columns); tissues=list(df.index)
    pos={order_idx[i]:i for i in range(len(order_idx))}
    total=0.0
    for t in tissues:
        w=weights.get(t,1.0)
        for i in range(len(order_idx)):
            for j in range(i+1,len(order_idx)):
                a,b=order_idx[i],order_idx[j]
                va,vb=df.at[t,drugs[a]],df.at[t,drugs[b]]
                if pd.isna(va) or pd.isna(vb): continue
                if vb>va: total+=w
    return total


In [2]:
# ============ TA-Mallows (approx) with SOFT DEADLINE + Benchmarks ============

def ta_mallows_order_soft(df: pd.DataFrame, lam: int,
                          max_iters: int = 2000, seed: int = 42,
                          time_budget_s: float = 3600.0):
    """
    Soft-deadline wrapper around your TA-Mallows (approx) local search.
    - Builds Borda-like start, computes tissue weights, does 2-swap hill climb.
    - Checks wall clock frequently; if budget exceeded, returns best-so-far
      along with the λ-consistent prefix from that order.
    """
    t0 = time.time()
    random.seed(seed)

    # Precompute majority-based tissue weights
    weights = tissue_weights_from_agreement(df)
    if time.time() - t0 > time_budget_s:
        return [], set(), time.time() - t0, "deadline"

    # Start order: mean efficacy descending (Borda-like)
    mean_vals = df.mean(axis=0).sort_values(ascending=False)
    order_idx = [df.columns.get_loc(c) for c in mean_vals.index.tolist()]
    best = order_idx[:]

    # Initial cost
    best_cost = kendall_disagreement(best, df, weights)

    # Hill climb (2-swap)
    for it in range(max_iters):
        # soft deadline check
        if time.time() - t0 > time_budget_s:
            # produce λ-prefix from best-so-far and return
            _, _, Ssets = pairwise_tissue_sets(df)
            path_idx, S_common, _ = path_intersection_prefix(best, Ssets, lam)
            drugs_list = list(df.columns)
            order = [drugs_list[i] for i in path_idx]
            return order, S_common, time.time() - t0, "deadline"

        i, j = sorted(random.sample(range(len(best)), 2))
        cand = best[:]
        cand[i], cand[j] = cand[j], cand[i]
        cost = kendall_disagreement(cand, df, weights)
        if cost < best_cost:
            best, best_cost = cand, cost

        # optional periodic check inside long runs
        if (it & 255) == 0 and (time.time() - t0 > time_budget_s):
            _, _, Ssets = pairwise_tissue_sets(df)
            path_idx, S_common, _ = path_intersection_prefix(best, Ssets, lam)
            drugs_list = list(df.columns)
            order = [drugs_list[i] for i in path_idx]
            return order, S_common, time.time() - t0, "deadline"

    # Finished within budget: return λ-prefix from final best order
    _, _, Ssets = pairwise_tissue_sets(df)
    path_idx, S_common, _ = path_intersection_prefix(best, Ssets, lam)
    drugs_list = list(df.columns)
    order = [drugs_list[i] for i in path_idx]
    return order, S_common, time.time() - t0, "ok"


# ---------------- Benchmark 1: λ-increasing on ALL drugs ----------------
def tamall_lambda_increasing(csv_path="efficacy.csv",
                             lam_start=5, lam_end=60, lam_step=5,
                             max_iters=2000, seed=42,
                             time_budget_s=3600,
                             out_xlsx="tamall_lambda_increasing.xlsx"):
    """
    Runs TA-Mallows (approx) on ALL drugs for increasing λ.
    Soft deadline per λ; saves Excel with full per-step results.
    """
    df_full = load_ctx_drug(csv_path)
    n_drugs = df_full.shape[1]
    rows = []
    print(f"TA-Mallows λ-sweep on ALL {n_drugs} drugs (soft deadline per λ = {time_budget_s//60:.0f} min)")

    for lam in range(lam_start, lam_end + 1, lam_step):
        order, S_common, elapsed, status = ta_mallows_order_soft(
            df_full, lam, max_iters=max_iters, seed=seed, time_budget_s=time_budget_s
        )
        rows.append({
            "method": "TA-Mallows(approx)",
            "n_drugs": n_drugs,
            "lambda": lam,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  λ={lam:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping λ sweep.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ------------- Benchmark 2: drugs-increasing (fixed λ), nested -------------
def tamall_drug_increasing(csv_path="efficacy.csv",
                           lam_fixed=30,
                           start_size=5, step=5, seed=42,
                           max_iters=2000,
                           time_budget_s=3600,
                           out_xlsx="tamall_drug_increasing.xlsx"):
    """
    Increases #drugs with fixed λ using nested, reproducible subsets (same seed).
    Soft deadline per size; saves Excel with per-step results.
    """
    df_full = load_ctx_drug(csv_path)
    n_total = df_full.shape[1]
    sizes = list(range(start_size, n_total + 1, step))
    rows = []

    # reproducible nested subset of columns
    rng = random.Random(seed)
    cols = list(df_full.columns)
    rng.shuffle(cols)

    print(f"TA-Mallows drug-sweep at λ={lam_fixed}; total drugs = {n_total} (soft deadline per size = {time_budget_s//60:.0f} min)")
    for n in sizes:
        sub = df_full[cols[:n]]
        order, S_common, elapsed, status = ta_mallows_order_soft(
            sub, lam_fixed, max_iters=max_iters, seed=seed, time_budget_s=time_budget_s
        )
        rows.append({
            "method": "TA-Mallows(approx)",
            "n_drugs": n,
            "lambda": lam_fixed,
            "path": " > ".join(order),
            "path_length": len(order),
            "exec_time_s": elapsed,
            "status": status
        })
        print(f"  n={n:3d} | status={status:8s} | time={elapsed:.2f} s | len={len(order)}")
        if status == "deadline":
            print("  ⏱️ Soft deadline hit; stopping size growth.")
            break

    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(out_xlsx) as xw:
        out_df.to_excel(xw, sheet_name="results", index=False)
    print(f"✅ Saved -> {out_xlsx}")
    return out_df


# ---------------------- Example calls (uncomment to run) ----------------------
# tamall_lambda_increasing("efficacy.csv",
#                          lam_start=3, lam_end=60, lam_step=3,
#                          max_iters=2000, seed=42,
#                          time_budget_s=3600,
#                          out_xlsx="tamall_lambda_increasing.xlsx")

# tamall_drug_increasing("efficacy.csv",
#                        lam_fixed=30,
#                        start_size=5, step=5, seed=42,
#                        max_iters=2000,
#                        time_budget_s=3600,
#                        out_xlsx="tamall_drug_increasing.xlsx")


In [ ]:
 tamall_drug_increasing("efficacy.csv",
                        lam_fixed=30,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m.xlsx")


TA-Mallows drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=9.93 s | len=3
  n= 10 | status=ok       | time=41.58 s | len=4
  n= 15 | status=ok       | time=96.13 s | len=4
  n= 20 | status=ok       | time=175.46 s | len=3
  n= 25 | status=ok       | time=277.50 s | len=3
  n= 30 | status=ok       | time=400.60 s | len=3
  n= 35 | status=ok       | time=555.98 s | len=2
  n= 40 | status=ok       | time=719.94 s | len=2
  n= 45 | status=ok       | time=910.91 s | len=1
  n= 50 | status=ok       | time=1114.31 s | len=1
  n= 55 | status=deadline | time=1201.78 s | len=1
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,30,MESALAMINE > VEMURAFENIB > NITAZOXANIDE,3,9.933298,ok
1,TA-Mallows(approx),10,30,THEOPHYLLINE > DEXAMETHASONE > PD-0166285 > ME...,4,41.584127,ok
2,TA-Mallows(approx),15,30,THEOPHYLLINE > DEXAMETHASONE > PD-0166285 > ME...,4,96.127474,ok
3,TA-Mallows(approx),20,30,THEOPHYLLINE > DEXAMETHASONE > PD-0166285,3,175.458918,ok
4,TA-Mallows(approx),25,30,THEOPHYLLINE > DEXAMETHASONE > PD-0166285,3,277.500617,ok
5,TA-Mallows(approx),30,30,THEOPHYLLINE > DEXAMETHASONE > PD-0166285,3,400.597605,ok
6,TA-Mallows(approx),35,30,THEOPHYLLINE > DEXAMETHASONE,2,555.980139,ok
7,TA-Mallows(approx),40,30,THEOPHYLLINE > DEXAMETHASONE,2,719.935816,ok
8,TA-Mallows(approx),45,30,THEOPHYLLINE,1,910.905031,ok
9,TA-Mallows(approx),50,30,THEOPHYLLINE,1,1114.314639,ok


In [3]:
# ===== HEADER: loader + helpers =====
import time, itertools, random
import pandas as pd
import numpy as np

def load_ctx_drug(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = df.index.astype(str).str.strip()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def pairwise_tissue_sets(df: pd.DataFrame):
    """Return drugs, tissues, and S[a][b] = set of tissues where a>b."""
    drugs = list(df.columns)
    tissues = list(df.index)
    n = len(drugs)
    S = [[set() for _ in range(n)] for __ in range(n)]
    for i, a in enumerate(drugs):
        for j, b in enumerate(drugs):
            if i==j: continue
            wins = set()
            for t in tissues:
                va, vb = df.at[t, a], df.at[t, b]
                if pd.isna(va) or pd.isna(vb):
                    continue
                if va > vb:
                    wins.add(t)
            S[i][j] = wins
    return drugs, tissues, S

def path_intersection_prefix(order_idxs, S, lam):
    """Given index order and S[i][j], return the longest λ-consistent prefix path and its common tissues S*."""
    if not order_idxs: return [], set(), []
    S_common = None
    adj_sets = []
    path = [order_idxs[0]]
    for i,j in zip(order_idxs[:-1], order_idxs[1:]):
        Sab = S[i][j]
        S_common = Sab if S_common is None else (S_common & Sab)
        if len(S_common) >= lam:
            path.append(j)
            adj_sets.append(Sab & S_common)
        else:
            break
    return path, (S_common if S_common is not None else set()), adj_sets

def print_result(tag, elapsed_s, order, S):
    print(f"[{tag}] time = {elapsed_s:.3f} s | path length = {len(order)} | |S| = {len(S)}")
    if order:
        print("  Path:", " > ".join(order))
    else:
        print("  Path: (empty)")


In [ ]:
tamall_drug_increasing("efficacy.csv",
                        lam_fixed=35,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasingl3530m.xlsx")


TA-Mallows drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.31 s | len=1
  n= 10 | status=ok       | time=23.78 s | len=2
  n= 15 | status=ok       | time=55.03 s | len=2
  n= 20 | status=ok       | time=99.63 s | len=2
  n= 25 | status=ok       | time=156.77 s | len=2
  n= 30 | status=ok       | time=228.00 s | len=2
  n= 35 | status=ok       | time=309.51 s | len=2
  n= 40 | status=ok       | time=407.45 s | len=2
  n= 45 | status=ok       | time=519.06 s | len=1
  n= 50 | status=ok       | time=638.15 s | len=1
  n= 55 | status=ok       | time=777.20 s | len=1
  n= 60 | status=ok       | time=927.46 s | len=1
  n= 65 | status=ok       | time=1089.42 s | len=1
  n= 70 | status=deadline | time=1201.79 s | len=1
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasingl3530m.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,35,MESALAMINE,1,5.308044,ok
1,TA-Mallows(approx),10,35,THEOPHYLLINE > DEXAMETHASONE,2,23.776231,ok
2,TA-Mallows(approx),15,35,THEOPHYLLINE > DEXAMETHASONE,2,55.033564,ok
3,TA-Mallows(approx),20,35,THEOPHYLLINE > DEXAMETHASONE,2,99.628491,ok
4,TA-Mallows(approx),25,35,THEOPHYLLINE > DEXAMETHASONE,2,156.771464,ok
5,TA-Mallows(approx),30,35,THEOPHYLLINE > DEXAMETHASONE,2,227.996328,ok
6,TA-Mallows(approx),35,35,THEOPHYLLINE > DEXAMETHASONE,2,309.507178,ok
7,TA-Mallows(approx),40,35,THEOPHYLLINE > DEXAMETHASONE,2,407.453848,ok
8,TA-Mallows(approx),45,35,THEOPHYLLINE,1,519.058007,ok
9,TA-Mallows(approx),50,35,THEOPHYLLINE,1,638.150548,ok


In [ ]:
tamall_drug_increasing("efficacy-diabities.csv",
                        lam_fixed=30,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diabities.xlsx")

TA-Mallows drug-sweep at λ=30; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=7.50 s | len=5
  n= 10 | status=ok       | time=36.67 s | len=3
  n= 15 | status=ok       | time=85.95 s | len=2
  n= 20 | status=ok       | time=154.33 s | len=3
  n= 25 | status=ok       | time=244.53 s | len=3
  n= 30 | status=ok       | time=351.80 s | len=3
  n= 35 | status=ok       | time=481.15 s | len=3
  n= 40 | status=ok       | time=627.16 s | len=4
  n= 45 | status=ok       | time=803.59 s | len=3
  n= 50 | status=ok       | time=983.64 s | len=3
  n= 55 | status=deadline | time=1202.21 s | len=3
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diabities.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,30,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,7.496075,ok
1,TA-Mallows(approx),10,30,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > WO...,3,36.668929,ok
2,TA-Mallows(approx),15,30,CYCLOPHOSPHAMIDE ANHYDROUS > OXIDOPAMINE HYDRO...,2,85.951415,ok
3,TA-Mallows(approx),20,30,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > O...,3,154.328330,ok
4,TA-Mallows(approx),25,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,3,244.526199,ok
5,TA-Mallows(approx),30,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,3,351.802849,ok
6,TA-Mallows(approx),35,30,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,3,481.146236,ok
7,TA-Mallows(approx),40,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,4,627.159967,ok
8,TA-Mallows(approx),45,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,3,803.593265,ok
9,TA-Mallows(approx),50,30,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,3,983.635569,ok


In [ ]:
tamall_drug_increasing("efficacy-diabities.csv",
                        lam_fixed=35,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diabities_l35.xlsx")

TA-Mallows drug-sweep at λ=35; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=8.08 s | len=3
  n= 10 | status=ok       | time=36.36 s | len=3
  n= 15 | status=ok       | time=89.41 s | len=1
  n= 20 | status=ok       | time=155.35 s | len=2
  n= 25 | status=ok       | time=243.20 s | len=3
  n= 30 | status=ok       | time=352.85 s | len=3
  n= 35 | status=ok       | time=480.58 s | len=3
  n= 40 | status=ok       | time=634.92 s | len=2
  n= 45 | status=ok       | time=801.55 s | len=2
  n= 50 | status=ok       | time=993.39 s | len=2
  n= 55 | status=deadline | time=1201.34 s | len=2
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diabities_l35.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,35,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,3,8.081879,ok
1,TA-Mallows(approx),10,35,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > WO...,3,36.356120,ok
2,TA-Mallows(approx),15,35,CYCLOPHOSPHAMIDE ANHYDROUS,1,89.406180,ok
3,TA-Mallows(approx),20,35,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS,2,155.346237,ok
4,TA-Mallows(approx),25,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,3,243.198796,ok
5,TA-Mallows(approx),30,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,3,352.850461,ok
6,TA-Mallows(approx),35,35,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,3,480.581030,ok
7,TA-Mallows(approx),40,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A,2,634.916544,ok
8,TA-Mallows(approx),45,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A,2,801.554710,ok
9,TA-Mallows(approx),50,35,CHEMBL:CHEMBL596633 > TRICHOSTATIN A,2,993.388999,ok


In [ ]:
tamall_drug_increasing("efficacy-diabities.csv",
                        lam_fixed=25,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diabities_l25.xlsx")

TA-Mallows drug-sweep at λ=25; total drugs = 273 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=8.88 s | len=5
  n= 10 | status=ok       | time=37.03 s | len=3
  n= 15 | status=ok       | time=84.89 s | len=2
  n= 20 | status=ok       | time=158.59 s | len=3
  n= 25 | status=ok       | time=242.78 s | len=4
  n= 30 | status=ok       | time=350.91 s | len=4
  n= 35 | status=ok       | time=483.41 s | len=4
  n= 40 | status=ok       | time=629.61 s | len=5
  n= 45 | status=ok       | time=810.91 s | len=4
  n= 50 | status=ok       | time=995.95 s | len=4
  n= 55 | status=deadline | time=1201.84 s | len=4
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diabities_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,25,CARBOPLATIN > CHEMBL:CHEMBL493863 > LEUCOVORIN...,5,8.884873,ok
1,TA-Mallows(approx),10,25,CYCLOPHOSPHAMIDE ANHYDROUS > DACTINOMYCIN > WO...,3,37.026851,ok
2,TA-Mallows(approx),15,25,CYCLOPHOSPHAMIDE ANHYDROUS > OXIDOPAMINE HYDRO...,2,84.885900,ok
3,TA-Mallows(approx),20,25,SULFASALAZINE > CYCLOPHOSPHAMIDE ANHYDROUS > O...,3,158.591460,ok
4,TA-Mallows(approx),25,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > CYCLOPHO...,4,242.782357,ok
5,TA-Mallows(approx),30,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,4,350.911748,ok
6,TA-Mallows(approx),35,25,CHEMBL:CHEMBL596633 > SULFASALAZINE > CHEMBL:C...,4,483.413843,ok
7,TA-Mallows(approx),40,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,5,629.614960,ok
8,TA-Mallows(approx),45,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,4,810.907280,ok
9,TA-Mallows(approx),50,25,CHEMBL:CHEMBL596633 > TRICHOSTATIN A > SULFASA...,4,995.950561,ok


In [5]:
tamall_drug_increasing("efficacy-bpco.csv",
                        lam_fixed=25,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_bpco_l25.xlsx")

TA-Mallows drug-sweep at λ=25; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.18 s | len=3
  n= 10 | status=ok       | time=24.48 s | len=2
  n= 15 | status=ok       | time=55.93 s | len=2
  n= 20 | status=ok       | time=102.07 s | len=3
  n= 25 | status=ok       | time=163.73 s | len=3
  n= 30 | status=ok       | time=229.20 s | len=1
  n= 35 | status=ok       | time=311.77 s | len=1
  n= 40 | status=ok       | time=410.70 s | len=1
  n= 45 | status=ok       | time=514.80 s | len=2
  n= 50 | status=ok       | time=637.97 s | len=2
  n= 55 | status=ok       | time=779.46 s | len=2
  n= 60 | status=ok       | time=937.54 s | len=3
  n= 65 | status=ok       | time=1085.20 s | len=3
  n= 70 | status=deadline | time=1201.41 s | len=3
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_bpco_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,25,OSIMERTINIB > SORAFENIB > FULVESTRANT,3,5.184848,ok
1,TA-Mallows(approx),10,25,THEOPHYLLINE > OSIMERTINIB,2,24.480952,ok
2,TA-Mallows(approx),15,25,THEOPHYLLINE > OLANZAPINE,2,55.928419,ok
3,TA-Mallows(approx),20,25,THEOPHYLLINE > OLANZAPINE > THALIDOMIDE,3,102.073087,ok
4,TA-Mallows(approx),25,25,THEOPHYLLINE > OLANZAPINE > THALIDOMIDE,3,163.729913,ok
5,TA-Mallows(approx),30,25,THEOPHYLLINE,1,229.198648,ok
6,TA-Mallows(approx),35,25,THEOPHYLLINE,1,311.769025,ok
7,TA-Mallows(approx),40,25,THEOPHYLLINE,1,410.702256,ok
8,TA-Mallows(approx),45,25,CLOZAPINE > THEOPHYLLINE,2,514.799919,ok
9,TA-Mallows(approx),50,25,CLOZAPINE > THEOPHYLLINE,2,637.968387,ok


In [6]:
tamall_drug_increasing("efficacy-bpco.csv",
                        lam_fixed=30,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_bpco_l30.xlsx")

TA-Mallows drug-sweep at λ=30; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.47 s | len=3
  n= 10 | status=ok       | time=23.83 s | len=2
  n= 15 | status=ok       | time=54.88 s | len=1
  n= 20 | status=ok       | time=99.94 s | len=1
  n= 25 | status=ok       | time=155.69 s | len=1
  n= 30 | status=ok       | time=227.65 s | len=1
  n= 35 | status=ok       | time=310.61 s | len=1
  n= 40 | status=ok       | time=405.63 s | len=1
  n= 45 | status=ok       | time=516.11 s | len=2
  n= 50 | status=ok       | time=637.23 s | len=2
  n= 55 | status=ok       | time=773.45 s | len=2
  n= 60 | status=ok       | time=918.34 s | len=3
  n= 65 | status=ok       | time=1084.44 s | len=2
  n= 70 | status=deadline | time=1201.67 s | len=2
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_bpco_l30.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,30,OSIMERTINIB > SORAFENIB > FULVESTRANT,3,5.467083,ok
1,TA-Mallows(approx),10,30,THEOPHYLLINE > OSIMERTINIB,2,23.825830,ok
2,TA-Mallows(approx),15,30,THEOPHYLLINE,1,54.876402,ok
3,TA-Mallows(approx),20,30,THEOPHYLLINE,1,99.937009,ok
4,TA-Mallows(approx),25,30,THEOPHYLLINE,1,155.689253,ok
5,TA-Mallows(approx),30,30,THEOPHYLLINE,1,227.649766,ok
6,TA-Mallows(approx),35,30,THEOPHYLLINE,1,310.614810,ok
7,TA-Mallows(approx),40,30,THEOPHYLLINE,1,405.634269,ok
8,TA-Mallows(approx),45,30,CLOZAPINE > THEOPHYLLINE,2,516.113146,ok
9,TA-Mallows(approx),50,30,CLOZAPINE > THEOPHYLLINE,2,637.230264,ok


In [7]:
tamall_drug_increasing("efficacy-bpco.csv",
                        lam_fixed=35,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_bpco_l35.xlsx")

TA-Mallows drug-sweep at λ=35; total drugs = 238 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.55 s | len=1
  n= 10 | status=ok       | time=23.40 s | len=2
  n= 15 | status=ok       | time=54.38 s | len=1
  n= 20 | status=ok       | time=97.95 s | len=1
  n= 25 | status=ok       | time=154.54 s | len=1
  n= 30 | status=ok       | time=224.75 s | len=1
  n= 35 | status=ok       | time=307.60 s | len=1
  n= 40 | status=ok       | time=402.76 s | len=1
  n= 45 | status=ok       | time=509.07 s | len=2
  n= 50 | status=ok       | time=633.16 s | len=2
  n= 55 | status=ok       | time=765.74 s | len=2
  n= 60 | status=ok       | time=914.54 s | len=3
  n= 65 | status=ok       | time=1082.47 s | len=2
  n= 70 | status=deadline | time=1201.99 s | len=2
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_bpco_l35.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,35,OSIMERTINIB,1,5.549181,ok
1,TA-Mallows(approx),10,35,THEOPHYLLINE > OSIMERTINIB,2,23.397879,ok
2,TA-Mallows(approx),15,35,THEOPHYLLINE,1,54.384918,ok
3,TA-Mallows(approx),20,35,THEOPHYLLINE,1,97.946421,ok
4,TA-Mallows(approx),25,35,THEOPHYLLINE,1,154.537818,ok
5,TA-Mallows(approx),30,35,THEOPHYLLINE,1,224.754648,ok
6,TA-Mallows(approx),35,35,THEOPHYLLINE,1,307.601855,ok
7,TA-Mallows(approx),40,35,THEOPHYLLINE,1,402.760694,ok
8,TA-Mallows(approx),45,35,CLOZAPINE > THEOPHYLLINE,2,509.071176,ok
9,TA-Mallows(approx),50,35,CLOZAPINE > THEOPHYLLINE,2,633.160417,ok


In [4]:
tamall_drug_increasing("efficacy-diabetes.csv",
                        lam_fixed=35,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diab_l35.xlsx")

TA-Mallows drug-sweep at λ=35; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=7.73 s | len=3
  n= 10 | status=ok       | time=26.95 s | len=3
  n= 15 | status=ok       | time=56.42 s | len=4
  n= 20 | status=ok       | time=102.15 s | len=4
  n= 25 | status=ok       | time=156.43 s | len=2
  n= 30 | status=ok       | time=230.79 s | len=2
  n= 35 | status=ok       | time=315.01 s | len=2
  n= 40 | status=ok       | time=423.33 s | len=2
  n= 45 | status=ok       | time=524.77 s | len=2
  n= 50 | status=ok       | time=642.35 s | len=2
  n= 55 | status=ok       | time=776.05 s | len=2
  n= 60 | status=ok       | time=928.94 s | len=2
  n= 65 | status=ok       | time=1100.90 s | len=2
  n= 70 | status=deadline | time=1201.35 s | len=2
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diab_l35.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,35,ATENOLOL > HUMAN > RG-1530,3,7.726717,ok
1,TA-Mallows(approx),10,35,ATENOLOL > HUMAN > RG-1530,3,26.948424,ok
2,TA-Mallows(approx),15,35,ASPIRIN > ATENOLOL > HUMAN > RG-1530,4,56.423544,ok
3,TA-Mallows(approx),20,35,ASPIRIN > ATENOLOL > HUMAN > GSK-269962A,4,102.153602,ok
4,TA-Mallows(approx),25,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,156.434115,ok
5,TA-Mallows(approx),30,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,230.789529,ok
6,TA-Mallows(approx),35,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,315.007638,ok
7,TA-Mallows(approx),40,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,423.326259,ok
8,TA-Mallows(approx),45,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,524.766561,ok
9,TA-Mallows(approx),50,35,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,642.348627,ok


In [5]:
tamall_drug_increasing("efficacy-diabetes.csv",
                        lam_fixed=30,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diab_l30.xlsx")

TA-Mallows drug-sweep at λ=30; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.68 s | len=3
  n= 10 | status=ok       | time=23.56 s | len=3
  n= 15 | status=ok       | time=54.91 s | len=4
  n= 20 | status=ok       | time=98.91 s | len=4
  n= 25 | status=ok       | time=156.33 s | len=2
  n= 30 | status=ok       | time=226.67 s | len=2
  n= 35 | status=ok       | time=310.54 s | len=2
  n= 40 | status=ok       | time=407.64 s | len=2
  n= 45 | status=ok       | time=518.88 s | len=2
  n= 50 | status=ok       | time=641.08 s | len=2
  n= 55 | status=ok       | time=776.88 s | len=2
  n= 60 | status=ok       | time=923.33 s | len=2
  n= 65 | status=ok       | time=1095.60 s | len=2
  n= 70 | status=deadline | time=1201.32 s | len=2
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diab_l30.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,30,ATENOLOL > HUMAN > RG-1530,3,5.680735,ok
1,TA-Mallows(approx),10,30,ATENOLOL > HUMAN > RG-1530,3,23.564621,ok
2,TA-Mallows(approx),15,30,ASPIRIN > ATENOLOL > HUMAN > RG-1530,4,54.905513,ok
3,TA-Mallows(approx),20,30,ASPIRIN > ATENOLOL > HUMAN > GSK-269962A,4,98.911147,ok
4,TA-Mallows(approx),25,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,156.325189,ok
5,TA-Mallows(approx),30,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,226.668168,ok
6,TA-Mallows(approx),35,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,310.542725,ok
7,TA-Mallows(approx),40,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,407.643803,ok
8,TA-Mallows(approx),45,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,518.879708,ok
9,TA-Mallows(approx),50,30,ASPIRIN > DOXEPIN HYDROCHLORIDE,2,641.075414,ok


In [6]:
tamall_drug_increasing("efficacy-diabetes.csv",
                        lam_fixed=25,
                        start_size=5, step=5, seed=42,
                        max_iters=2000,
                        time_budget_s=1200,
                        out_xlsx="tamall_drug_increasing30m_diab_l25.xlsx")

TA-Mallows drug-sweep at λ=25; total drugs = 171 (soft deadline per size = 20 min)
  n=  5 | status=ok       | time=5.04 s | len=3
  n= 10 | status=ok       | time=23.65 s | len=3
  n= 15 | status=ok       | time=55.25 s | len=4
  n= 20 | status=ok       | time=99.63 s | len=4
  n= 25 | status=ok       | time=158.43 s | len=3
  n= 30 | status=ok       | time=229.32 s | len=3
  n= 35 | status=ok       | time=312.18 s | len=3
  n= 40 | status=ok       | time=415.57 s | len=3
  n= 45 | status=ok       | time=518.97 s | len=3
  n= 50 | status=ok       | time=640.35 s | len=3
  n= 55 | status=ok       | time=785.21 s | len=3
  n= 60 | status=ok       | time=923.27 s | len=3
  n= 65 | status=ok       | time=1086.34 s | len=3
  n= 70 | status=deadline | time=1201.23 s | len=3
  ⏱️ Soft deadline hit; stopping size growth.
✅ Saved -> tamall_drug_increasing30m_diab_l25.xlsx


,method,n_drugs,lambda,path,path_length,exec_time_s,status
0,TA-Mallows(approx),5,25,ATENOLOL > HUMAN > RG-1530,3,5.036016,ok
1,TA-Mallows(approx),10,25,ATENOLOL > HUMAN > RG-1530,3,23.651906,ok
2,TA-Mallows(approx),15,25,ASPIRIN > ATENOLOL > HUMAN > RG-1530,4,55.252358,ok
3,TA-Mallows(approx),20,25,ASPIRIN > ATENOLOL > HUMAN > GSK-269962A,4,99.634196,ok
4,TA-Mallows(approx),25,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,158.425508,ok
5,TA-Mallows(approx),30,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,229.315638,ok
6,TA-Mallows(approx),35,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,312.184809,ok
7,TA-Mallows(approx),40,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,415.567022,ok
8,TA-Mallows(approx),45,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,518.965258,ok
9,TA-Mallows(approx),50,25,ASPIRIN > DOXEPIN HYDROCHLORIDE > ATENOLOL,3,640.346843,ok
